# 🚦 Traffic Signal Optimization — EDA & Model Analysis
**Summer 2024 | ML · Python · SQL · Random Forest · Feature Engineering**

This notebook walks through:
1. Exploratory Data Analysis (EDA)
2. Feature Engineering walkthrough
3. Model training & evaluation
4. Feature importance & interpretability
5. Signal timing recommendations

In [ ]:
import sys, os
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import r2_score

from src.utils.data_generator import generate_traffic_dataset
from src.pipeline.preprocessor import TrafficPreprocessor
from src.pipeline.model import TrafficSignalModel

plt.style.use('dark_background')
sns.set_palette('viridis')

print('Libraries loaded ✓')

## 1. Load & Inspect Dataset

In [ ]:
df = generate_traffic_dataset(n_records=55000)
print(f'Shape: {df.shape}')
df.head()

In [ ]:
df.describe().round(2)

## 2. Traffic Volume Patterns

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

hourly = df.groupby('hour')['vehicle_count'].mean()
axes[0].plot(hourly.index, hourly.values, color='#00d4aa', linewidth=2.5)
axes[0].fill_between(hourly.index, hourly.values, alpha=0.2, color='#00d4aa')
axes[0].set_title('Average Vehicle Count by Hour', fontsize=13)
axes[0].set_xlabel('Hour of Day')
axes[0].set_ylabel('Avg Vehicle Count')

type_vol = df.groupby('intersection_type')['vehicle_count'].mean()
axes[1].bar(type_vol.index, type_vol.values, color=['#00d4aa','#ff6b9d','#ffd93d'])
axes[1].set_title('Avg Volume by Intersection Type', fontsize=13)
axes[1].set_ylabel('Avg Vehicle Count')

plt.tight_layout()
plt.show()

## 3. Feature Engineering

In [ ]:
prep = TrafficPreprocessor()
X_train, X_test, y_train, y_test = prep.fit_transform(df)
print(f'Features: {len(prep.feature_columns)}')
print('Engineered features:', [f for f in prep.feature_columns if any(k in f for k in ['sin','cos','rush','congestion','per_lane'])])

## 4. Model Training & Evaluation

In [ ]:
model = TrafficSignalModel()
model.train(X_train, y_train, feature_names=prep.feature_columns)
metrics = model.evaluate(X_test, y_test, X_train, y_train)

print(f"\n✅ R² Score: {metrics['r2_score']}")
print(f"✅ MAE: {metrics['mae_seconds']} seconds")

## 5. Feature Importance

In [ ]:
fi = model.feature_importance_df()
top15 = fi.head(15)

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(top15['feature'][::-1], top15['importance_pct'][::-1],
               color='#00d4aa', alpha=0.85)
ax.set_xlabel('Importance %')
ax.set_title('Top 15 Feature Importances', fontsize=14)
plt.tight_layout()
plt.show()

## 6. Actual vs Predicted

In [ ]:
y_pred = model.predict(X_test)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(y_test[:2000], y_pred[:2000], alpha=0.3, s=5, color='#00d4aa')
mn, mx = y_test.min(), y_test.max()
axes[0].plot([mn,mx],[mn,mx],'--', color='#ff6b9d', linewidth=2)
axes[0].set_title('Actual vs Predicted Green Phase (s)', fontsize=13)
axes[0].set_xlabel('Actual'); axes[0].set_ylabel('Predicted')

residuals = y_test - y_pred
axes[1].hist(residuals, bins=60, color='#00d4aa', alpha=0.8, edgecolor='none')
axes[1].axvline(0, color='#ff6b9d', linestyle='--', linewidth=2)
axes[1].set_title('Residual Distribution', fontsize=13)
axes[1].set_xlabel('Residual (s)')

plt.tight_layout()
plt.show()
print(f'R² = {r2_score(y_test, y_pred):.4f}')